# Experiment harness

A **self-contained** notebook for improving the two pipelines *properly*. It does
not use the baked artifacts (`tfidf_artifacts/`, `models_out/`) — it rebuilds
features from scratch so every lever (including ingestion params like TF-IDF
n-grams) can be varied and the vectorizer is re-fit **inside each CV fold** (no
leakage). The only shared code is the CSV loader in `dataset.py`.

### Method (why you can trust the numbers)
- The eval set is tiny (20 rows), so a single eval score is mostly noise.
  Instead we use **repeated stratified k-fold cross-validation** on *all* the
  data and report **mean ± std** across folds. A change only "wins" if its gain
  clears the noise band.
- We change **one lever at a time** and log every result into a table.
- We keep a small **held-out test set** untouched until the very end.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    cross_validate, cross_val_predict, RepeatedStratifiedKFold,
    StratifiedKFold, RandomizedSearchCV, train_test_split,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import precision_recall_curve, f1_score, accuracy_score, classification_report
from lightgbm import LGBMClassifier
from sentence_transformers import SentenceTransformer

from dataset import load_dataset, TRAINING_CSV, EVAL_CSV, LABELS, ROOT

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")

## Load all the data

We combine the training and eval splits into one pool, then carve off a small
**held-out test set** we won't look at until the end. All experimentation happens
on `X_pool` / `y_pool` via cross-validation.

In [ ]:
train = load_dataset(TRAINING_CSV)
ev = load_dataset(EVAL_CSV)
texts  = np.array(train.texts + ev.texts)
labels = np.array(train.labels + ev.labels)
print(f"Total rows: {len(texts)}  |  class counts: {dict(zip(*np.unique(labels, return_counts=True)))}")

# Held-out test set (touched only at the very end).
X_pool, X_test, y_pool, y_test = train_test_split(
    texts, labels, test_size=0.2, stratify=labels, random_state=RANDOM_STATE
)
print(f"Experiment pool: {len(X_pool)}   Held-out test: {len(X_test)}")

## Embed once with BGE

BGE is pre-trained and never fits to *our* labels, so embedding all rows up front
is **not** leakage — and it makes the CV loops fast. The TF-IDF pipeline is
different: its vocabulary *does* learn from data, so we never pre-compute it; we
wrap it in a `Pipeline` so it re-fits inside every fold.

In [ ]:
embedder = SentenceTransformer(str(ROOT / "models" / "bge-small-en-v1.5"))
Xb_pool = embedder.encode(list(X_pool))   # BGE feature matrix for the pool
Xb_test = embedder.encode(list(X_test))
print("BGE embeddings:", Xb_pool.shape)

## The evaluation harness

`score()` runs repeated stratified 5-fold CV (×3 repeats = 15 fits) and records
mean ± std for accuracy and macro-F1 into a shared results table. Macro-F1 is our
primary metric (robust on small, roughly balanced data).

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
results = []

def score(name, estimator, X, y):
    r = cross_validate(estimator, X, y, cv=cv, scoring=["accuracy", "f1_macro"], n_jobs=1)
    row = dict(
        experiment=name,
        f1_macro=r["test_f1_macro"].mean(), f1_std=r["test_f1_macro"].std(),
        accuracy=r["test_accuracy"].mean(), acc_std=r["test_accuracy"].std(),
    )
    results.append(row)
    print(f"{name:38s}  F1={row['f1_macro']:.3f} +/- {row['f1_std']:.3f}   acc={row['accuracy']:.3f}")
    return row

# LightGBM base settings that work on this tiny sparse data (see classify_lightgbm.py).
def lgbm(**kw):
    params = dict(random_state=RANDOM_STATE, min_data_in_bin=1, min_child_samples=5, verbose=-1)
    params.update(kw)
    return LGBMClassifier(**params)

## Baselines

Reproduce the two current pipelines as the reference point.

In [ ]:
score("BGE + LogReg (baseline)", LogisticRegression(max_iter=1000), Xb_pool, y_pool)
score("TFIDF + LightGBM (baseline)",
      make_pipeline(TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True), lgbm()),
      X_pool, y_pool)

## Lever 1 — Logistic Regression regularization `C` + class weight

Sweep the main LogReg knob and try balanced class weights. This is the single
most useful knob for the BGE pipeline.

In [ ]:
for C in [0.01, 0.1, 1, 10, 100]:
    score(f"BGE + LogReg C={C}", LogisticRegression(C=C, max_iter=1000), Xb_pool, y_pool)
score("BGE + LogReg C=10 balanced",
      LogisticRegression(C=10, class_weight="balanced", max_iter=1000), Xb_pool, y_pool)

## Lever 2 — TF-IDF feature choices

Word vs. word+char n-grams and `min_df`. Char n-grams catch obfuscated toxicity
("id10t", "sh!t"). The vectorizer re-fits inside each fold automatically.

In [ ]:
score("TFIDF words(1,1) + LGBM",
      make_pipeline(TfidfVectorizer(stop_words="english", ngram_range=(1, 1), sublinear_tf=True), lgbm()),
      X_pool, y_pool)
score("TFIDF words(1,2) min_df=2 + LGBM",
      make_pipeline(TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2, sublinear_tf=True), lgbm()),
      X_pool, y_pool)
score("TFIDF char(2,5) + LGBM",
      make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True), lgbm()),
      X_pool, y_pool)

## Lever 3 — LightGBM hyperparameter search

A small randomized search over the tree knobs, cross-validated. `n_iter` is kept
low because the dataset is tiny.

In [ ]:
pipe = make_pipeline(
    TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True),
    lgbm(),
)
param_dist = {
    "lgbmclassifier__n_estimators": [50, 100, 200],
    "lgbmclassifier__learning_rate": [0.02, 0.05, 0.1],
    "lgbmclassifier__num_leaves": [7, 15, 31],
    "lgbmclassifier__reg_lambda": [0.0, 1.0, 5.0],
    "lgbmclassifier__class_weight": [None, "balanced"],
}
search = RandomizedSearchCV(
    pipe, param_dist, n_iter=15, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    scoring="f1_macro", random_state=RANDOM_STATE, n_jobs=1,
)
search.fit(X_pool, y_pool)
print("best f1_macro:", round(search.best_score_, 3))
print("best params:", {k.split('__')[1]: v for k, v in search.best_params_.items()})
score("TFIDF + LGBM (tuned)", search.best_estimator_, X_pool, y_pool)

## Leaderboard

Every experiment so far, ranked by cross-validated macro-F1. Only trust a
difference that is larger than the `f1_std` noise.

In [ ]:
board = pd.DataFrame(results).sort_values("f1_macro", ascending=False).reset_index(drop=True)
board

## Threshold tuning

The models output a *probability* of toxic; the default cutoff is 0.5. For
toxicity you often want **high recall on the toxic class** (don't miss abuse).
Here we get out-of-fold probabilities for the best BGE model and see how
precision/recall for `toxic` trade off as we move the threshold.

In [ ]:
best_bge = LogisticRegression(C=10, class_weight="balanced", max_iter=1000)
proba = cross_val_predict(best_bge, Xb_pool, y_pool,
                          cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
                          method="predict_proba")
toxic_col = list(best_bge.fit(Xb_pool, y_pool).classes_).index("toxic")
y_toxic = (y_pool == "toxic").astype(int)
prec, rec, thr = precision_recall_curve(y_toxic, proba[:, toxic_col])

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(thr, prec[:-1], label="precision (toxic)")
ax.plot(thr, rec[:-1], label="recall (toxic)")
ax.axvline(0.5, color="gray", ls="--", lw=1, label="default 0.5")
ax.set_xlabel("threshold for calling a comment 'toxic'"); ax.set_ylabel("score")
ax.set_title("Precision / recall vs. threshold (BGE + LogReg)"); ax.legend()
plt.show()

print("threshold  precision  recall")
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred = (proba[:, toxic_col] >= t)
    tp = (pred & (y_toxic == 1)).sum(); fp = (pred & (y_toxic == 0)).sum(); fn = (~pred & (y_toxic == 1)).sum()
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    print(f"   {t:.1f}      {p:.2f}      {r:.2f}")

## Ensemble the two pipelines

Average the toxic-probabilities from both models (a "soft vote"). Because the
pipelines see the data very differently (dense semantics vs. sparse surface
words), averaging often beats either alone.

In [ ]:
skf = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

# Out-of-fold toxic-probabilities from each pipeline (same folds -> comparable).
lr = LogisticRegression(C=10, class_weight="balanced", max_iter=1000)
lr_col = list(lr.fit(Xb_pool, y_pool).classes_).index("toxic")
p_bge = cross_val_predict(lr, Xb_pool, y_pool, cv=skf, method="predict_proba")[:, lr_col]

tfidf_pipe = make_pipeline(TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True), lgbm())
tf_col = list(tfidf_pipe.fit(X_pool, y_pool).classes_).index("toxic")
p_tfidf = cross_val_predict(tfidf_pipe, X_pool, y_pool, cv=skf, method="predict_proba")[:, tf_col]

for name, p in [("BGE only", p_bge), ("TFIDF only", p_tfidf), ("Ensemble (avg)", (p_bge + p_tfidf) / 2)]:
    pred = np.where(p >= 0.5, "toxic", "non_toxic")
    print(f"{name:16s}  F1={f1_score(y_pool, pred, pos_label='toxic'):.3f}  acc={accuracy_score(y_pool, pred):.3f}")

## Final check on the held-out test set

Only now do we touch `X_test`. This is the honest estimate of how the chosen
configuration generalizes. (Still only ~20 rows, so read it as a sanity check.)

In [ ]:
final = LogisticRegression(C=10, class_weight="balanced", max_iter=1000).fit(Xb_pool, y_pool)
print("Held-out test performance (BGE + LogReg C=10 balanced):\n")
print(classification_report(y_test, final.predict(Xb_test), zero_division=0))

## How to use this harness going forward

- **Add a lever** = add one `score("name", estimator, X, y)` call. It lands in the leaderboard automatically.
- **Trust the noise band**: a change is real only if the F1 gain > `f1_std`.
- **The biggest lever isn't here**: more and cleaner labeled data. When you add rows, just re-run top-to-bottom.
- Keep the held-out test **sacred** — look at it rarely, and never tune against it.